# GOLDS Notebook 02 — Environment Pipeline

This notebook walks through the full environment pipeline used by GOLDS (Game-Oriented Learning with Deep RL Systems), from the raw Gymnasium interface to vectorized, preprocessed environments ready for PPO training.

**Prerequisites:** You should be comfortable with Python, NumPy, and basic ML concepts (supervised learning, neural networks). No prior RL knowledge is required.

**What you will learn:**
1. How the Gymnasium `reset()`/`step()` loop works
2. Why Atari frames need heavy preprocessing before a neural network can learn from them
3. How `stable-retro` extends Gymnasium to ROM-based console games
4. How vectorized environments speed up training
5. How GOLDS implements self-play for two-player fighting games

In [ ]:
# Standard imports used throughout the notebook
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 120

---
## 1. The Gymnasium Interface

Every RL environment in this project follows the **Gymnasium** API (the maintained fork of OpenAI Gym). The interaction is deceptively simple — the agent and environment take turns:

```
Agent                     Environment
  |                           |
  |-------- reset() --------->|
  |<--- (obs, info) ----------|
  |                           |
  |--- step(action) --------->|
  |<-- (obs, reward,          |
  |     terminated,           |
  |     truncated, info) -----|
  |                           |
  |--- step(action) --------->|
  |         ...               |
```

### Key concepts

| Term | Meaning |
|------|--------|
| **Observation** | What the agent sees (e.g., an image of the game screen) |
| **Action** | What the agent does (e.g., press "right" + "jump") |
| **Reward** | Scalar feedback signal. The agent tries to maximize cumulative reward. |
| **Terminated** | `True` when the episode ends naturally (game over, goal reached). |
| **Truncated** | `True` when the episode is cut short by a time limit. |

The agent's goal in RL is to learn a **policy** $\pi(a \mid s)$ — a mapping from observations to actions — that maximizes the **expected discounted return**:

$$G_t = \sum_{k=0}^{\infty} \gamma^k \, r_{t+k+1}$$

where $\gamma \in [0, 1)$ is the **discount factor** that makes near-term rewards more valuable than distant ones.

### 1.1 Observation & Action Spaces

Every Gymnasium environment declares two `Space` objects:

- `env.observation_space` — describes what observations look like  
- `env.action_space` — describes what actions are valid

Common space types:

| Space | Example | Description |
|-------|---------|------------|
| `Box(low, high, shape)` | Images, continuous controls | Real-valued tensor with bounds |
| `Discrete(n)` | Atari actions (0..17) | Single integer from `{0, 1, ..., n-1}` |
| `MultiBinary(n)` | Retro button presses | Binary vector of length `n` (each button on/off) |

In [ ]:
import gymnasium as gym

# CartPole is a simple, always-available environment — perfect for showing the API.
env = gym.make("CartPole-v1", render_mode="rgb_array")

print("Observation space:", env.observation_space)
print("  shape:", env.observation_space.shape)
print("  dtype:", env.observation_space.dtype)
print()
print("Action space:", env.action_space)
print("  n:", env.action_space.n)

In [ ]:
# The core loop: reset → step → step → ... → done → reset
obs, info = env.reset(seed=42)
print(f"Initial observation: {obs}")
print(f"Info dict: {info}")
print()

total_reward = 0.0
steps = 0

for _ in range(500):  # safety limit
    action = env.action_space.sample()  # random policy
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    
    if terminated or truncated:
        break

print(f"Episode finished after {steps} steps with total reward {total_reward}")
env.close()

### 1.2 Visual Environments — What the Agent "Sees"

For Atari and retro games, the observation is a raw RGB image. Let's look at what that actually means for the agent.

In [ ]:
try:
    # Atari env — requires ale-py and the ROM
    atari_env = gym.make("ALE/Breakout-v5", render_mode="rgb_array")
    obs, info = atari_env.reset(seed=42)

    print(f"Observation space: {atari_env.observation_space}")
    print(f"Observation shape: {obs.shape}  dtype: {obs.dtype}")
    print(f"Action space: {atari_env.action_space} (Discrete, n={atari_env.action_space.n})")
    print(f"Action meanings: {atari_env.unwrapped.get_action_meanings()}")

    fig, ax = plt.subplots(1, 1, figsize=(4, 5))
    ax.imshow(obs)
    ax.set_title(f"Raw Breakout frame\n{obs.shape[0]}×{obs.shape[1]} px, {obs.shape[2]} channels")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    atari_env.close()
except Exception as e:
    print(f"Could not create Atari env (expected if ROMs not installed): {e}")
    print("\nThe raw observation would be a (210, 160, 3) uint8 array — a 210×160 RGB image.")
    print("That's 100,800 input values per frame! Preprocessing shrinks this dramatically.")

---
## 2. The Atari Preprocessing Stack

Raw Atari frames are 210×160 RGB images — that is **100,800 input values** per frame. A neural network *could* learn from these, but it would be slow and wasteful. The DeepMind DQN paper (Mnih et al., 2015) introduced a preprocessing pipeline that has become the de facto standard. Each wrapper in the stack addresses a specific problem.

The wrappers are applied in this order:

```
Raw Atari Env
  └─ NoopResetEnv        (diverse starting states)
      └─ MaxAndSkipEnv   (temporal downsampling + flicker removal)
          └─ EpisodicLifeEnv  (better value estimation)
              └─ FireResetEnv  (auto-start games)
                  └─ WarpFrame (84×84 grayscale)
                      └─ ClipRewardEnv (normalized rewards)
```

Let's look at each one and understand **what** it does and **why**.

### 2.1 NoopResetEnv — Diverse Starting States

**Problem:** Without intervention, every episode starts from the exact same game state. The agent can memorize an optimal sequence of actions rather than learning general strategies.

**Solution:** On reset, execute a random number of "no-op" (do nothing) actions — typically 1 to 30. This advances the game to a slightly different starting state each time.

```python
class NoopResetEnv(gym.Wrapper):
    def reset(self, **kwargs):
        self.env.reset(**kwargs)
        noops = np.random.randint(1, self.noop_max + 1)
        for _ in range(noops):
            obs, _, terminated, truncated, info = self.env.step(0)  # action 0 = NOOP
            if terminated or truncated:
                obs, info = self.env.reset(**kwargs)
        return obs, info
```

**Key insight:** This is a form of **domain randomization** — it forces the agent to handle slight variations in initial conditions.

### 2.2 MaxAndSkipEnv — Frame Skipping with Flicker Removal

**Problem 1 — Speed:** The Atari runs at 60 fps. Most actions don't need to change every 1/60th of a second. Processing all frames wastes computation.

**Problem 2 — Flicker:** The Atari 2600 hardware could only render a limited number of sprites per scanline. Games worked around this by alternating which sprites were drawn on odd vs. even frames. This means some objects are literally invisible on certain frames.

**Solution:** Repeat each action for `skip=4` frames. Return the pixel-wise **maximum** of the last two frames (to capture any flickering objects).

$$\text{obs}_t = \max\bigl(\text{frame}_{t-1},\; \text{frame}_{t}\bigr)$$

This gives us a 4x speedup in wall-clock training time, since the agent only needs to choose an action every 4th frame.

```python
class MaxAndSkipEnv(gym.Wrapper):
    def step(self, action):
        total_reward = 0.0
        for i in range(self._skip):   # repeat action 4 times
            obs, reward, terminated, truncated, info = self.env.step(action)
            if i == self._skip - 2: self._obs_buffer[0] = obs  # 2nd-to-last frame
            if i == self._skip - 1: self._obs_buffer[1] = obs  # last frame
            total_reward += reward
            if terminated or truncated: break
        return self._obs_buffer.max(axis=0), total_reward, ...  # pixel-wise max
```

In [ ]:
# Demonstrate max-pooling on synthetic "flickering" frames
np.random.seed(42)

# Simulate two consecutive 8x8 grayscale frames with a flickering sprite
frame_a = np.zeros((8, 8), dtype=np.uint8)
frame_b = np.zeros((8, 8), dtype=np.uint8)

# Object appears on frame A (rows 2-4, cols 2-4)
frame_a[2:5, 2:5] = 200

# Different object appears on frame B (rows 3-5, cols 5-7) — the sprite flickered
frame_b[3:6, 5:8] = 180

# Max pool captures both
max_frame = np.maximum(frame_a, frame_b)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, img, title in zip(axes, [frame_a, frame_b, max_frame],
                           ["Frame t-1\n(sprite A visible)",
                            "Frame t\n(sprite B visible)",
                            "max(t-1, t)\n(both captured)"]):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
plt.suptitle("MaxAndSkipEnv: pixel-wise maximum eliminates flicker", fontweight="bold")
plt.tight_layout()
plt.show()

### 2.3 EpisodicLifeEnv — Treating Each Life as an Episode

**Problem:** In games like Breakout, the agent has 5 lives. Without this wrapper, losing a life provides no learning signal — the episode only ends when *all* lives are gone. This makes it hard for the value function to learn that losing a life is bad.

**Solution:** Signal `terminated=True` whenever a life is lost. Crucially, the environment is *not* fully reset — only a no-op step is taken to advance past the death animation. The true reset only happens on game over.

This helps the **value function** $V(s)$ learn that states near death are low-value:

$$V(s) = \mathbb{E}_{\pi}\left[\sum_{k=0}^{\infty} \gamma^k r_{t+k+1} \;\middle|\; s_t = s\right]$$

Without life-as-episode, $V(s)$ has trouble distinguishing "about to die with 3 lives left" from "about to die on last life" because the episode reward structure is identical up to that point.

> **Important:** During *evaluation*, you typically disable this wrapper (`terminal_on_life_loss=False`) so that reported episode rewards reflect full games, not individual lives. GOLDS handles this in `EnvironmentFactory.create_eval_env()`.

### 2.4 FireResetEnv — Auto-Starting Games

**Problem:** Some Atari games (like Breakout) require pressing the FIRE button to start playing after a reset or death. Without this, the agent wastes time learning that it needs to press FIRE — a trivially uninteresting action.

**Solution:** Automatically press FIRE (and then a neutral action) after every reset.

```python
class FireResetEnv(gym.Wrapper):
    def reset(self, **kwargs):
        self.env.reset(**kwargs)
        obs, _, terminated, truncated, _ = self.env.step(1)  # FIRE
        if terminated or truncated:
            self.env.reset(**kwargs)
        obs, _, terminated, truncated, _ = self.env.step(2)  # neutral
        if terminated or truncated:
            self.env.reset(**kwargs)
        return obs, {}
```

This wrapper is only applied when FIRE is in the environment's action meanings.

### 2.5 WarpFrame — Resize and Grayscale

**Problem:** The raw 210×160×3 RGB image is too large and contains information the agent doesn't need (color is mostly irrelevant for gameplay).

**Solution:** Convert to grayscale and resize to 84×84×1.

| | Raw | Warped |
|---|---|---|
| Shape | (210, 160, 3) | (84, 84, 1) |
| Values per frame | 100,800 | 7,056 |
| Reduction | — | **93% fewer** |

The resizing uses `cv2.INTER_AREA` interpolation, which is best for downscaling (it averages over pixel neighborhoods rather than sampling individual pixels, reducing aliasing).

In [ ]:
try:
    import cv2
    
    # Demonstrate the WarpFrame transformation on a synthetic game-like image
    # Create a colorful 210x160 test image simulating a game screen
    raw = np.zeros((210, 160, 3), dtype=np.uint8)
    raw[0:30, :] = [50, 50, 50]       # score area (top)
    raw[30:180, :] = [20, 20, 80]     # play field (blue-ish)
    raw[180:, :] = [80, 40, 20]       # ground
    raw[100:120, 60:80] = [200, 200, 0]  # "player" sprite
    raw[50:60, 100:130] = [200, 0, 0]    # "enemy" sprite

    # Apply the same transforms as WarpFrame
    gray = cv2.cvtColor(raw, cv2.COLOR_RGB2GRAY)
    warped = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(raw)
    axes[0].set_title(f"Raw RGB\n{raw.shape}")
    axes[1].imshow(gray, cmap="gray")
    axes[1].set_title(f"Grayscale\n{gray.shape}")
    axes[2].imshow(warped, cmap="gray")
    axes[2].set_title(f"Warped 84×84\n{warped.shape}")
    for ax in axes:
        ax.axis("off")
    plt.suptitle("WarpFrame: RGB → Grayscale → 84×84", fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"Raw values per frame:    {raw.size:>10,}")
    print(f"Warped values per frame: {warped.size:>10,}")
    print(f"Reduction:               {1 - warped.size / raw.size:>10.1%}")
except ImportError:
    print("OpenCV not installed. Install with: pip install opencv-python")

### 2.6 ClipRewardEnv — Reward Normalization

**Problem:** Different Atari games have wildly different reward scales. In Pong, scoring gives +1 or -1. In Space Invaders, destroying enemies gives 5, 10, 15, or 30 points. This makes it hard to use the same hyperparameters across games.

**Solution:** Clip all rewards to $\{-1, 0, +1\}$ using the sign function:

$$r_{\text{clipped}} = \text{sign}(r) = \begin{cases} +1 & \text{if } r > 0 \\ 0 & \text{if } r = 0 \\ -1 & \text{if } r < 0 \end{cases}$$

**Trade-off:** The agent no longer distinguishes between getting 10 points and 100 points — both become +1. This simplifies learning but loses some information about the relative value of different events. For GOLDS, this trade-off is acceptable because we care about *winning the game*, not maximizing raw score.

In [ ]:
# Visualize reward clipping
raw_rewards = np.array([-30, -10, -1, 0, 0, 0, 5, 10, 15, 30, 0, -5, 100])
clipped_rewards = np.sign(raw_rewards)

fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=False)

axes[0].bar(range(len(raw_rewards)), raw_rewards, color=["salmon" if r < 0 else "skyblue" if r > 0 else "lightgray" for r in raw_rewards])
axes[0].set_title("Raw rewards (varying scale)")
axes[0].set_ylabel("Reward")
axes[0].set_xlabel("Time step")

axes[1].bar(range(len(clipped_rewards)), clipped_rewards, color=["salmon" if r < 0 else "skyblue" if r > 0 else "lightgray" for r in clipped_rewards])
axes[1].set_title("Clipped rewards (sign only)")
axes[1].set_ylabel("Reward")
axes[1].set_xlabel("Time step")
axes[1].set_ylim(-1.5, 1.5)

plt.suptitle("ClipRewardEnv: sign(r) normalizes across games", fontweight="bold")
plt.tight_layout()
plt.show()

### 2.7 Putting It All Together — The Full Atari Stack

In Stable-Baselines3 (and GOLDS for Atari games), all these wrappers are combined in the `AtariWrapper` class:

```python
class AtariWrapper(gym.Wrapper):
    def __init__(self, env, noop_max=30, frame_skip=4,
                 screen_size=84, terminal_on_life_loss=True,
                 clip_reward=True):
        if noop_max > 0:
            env = NoopResetEnv(env, noop_max=noop_max)
        if frame_skip > 1:
            env = MaxAndSkipEnv(env, skip=frame_skip)
        if terminal_on_life_loss:
            env = EpisodicLifeEnv(env)
        if "FIRE" in env.unwrapped.get_action_meanings():
            env = FireResetEnv(env)
        env = WarpFrame(env, width=screen_size, height=screen_size)
        if clip_reward:
            env = ClipRewardEnv(env)
        super().__init__(env)
```

After this pipeline, what was a `(210, 160, 3)` RGB image becomes a `(84, 84, 1)` grayscale frame — a **93% reduction** in input size.

---
## 3. Retro Games via `stable-retro`

While the Atari wrappers above work for the Atari 2600 (via `ale-py`), GOLDS also trains agents on **retro console games** — NES, SNES, Genesis, and more — using the `stable-retro` library. This opens up classics like Super Mario Bros, Sonic the Hedgehog, and Mortal Kombat II.

### 3.1 How Retro Differs from Atari

| Feature | Atari (ale-py) | Retro (stable-retro) |
|---------|---------------|---------------------|
| Hardware emulated | Atari 2600 | NES, SNES, Genesis, GBA, ... |
| ROM source | Bundled with ale-py | User must import ROMs |
| Starting states | Noop randomization | **Save states** (.state files) |
| Reward signal | Built into emulator | **data.json** — user-defined memory addresses |
| Action space | `Discrete(n)` | `MultiBinary(n)` — individual button presses |
| Resolution | Fixed 210×160 | Varies by console (256×224, 320×224, ...) |

### 3.2 ROM-Based Architecture

Retro environments need three things:

1. **The ROM file** — the actual game cartridge data (not distributed with the library for legal reasons)
2. **Save states** (`.state` files) — frozen snapshots of the emulator state that let you start from specific levels/situations
3. **`data.json`** — a mapping from **RAM addresses** to game variables (score, lives, health, timer) used to compute rewards

Example `data.json` structure (simplified):
```json
{
  "info": {
    "score": {"address": 1984, "type": "|u4"},
    "lives": {"address": 1882, "type": "|u1"}
  }
}
```

A companion **`scenario.json`** then defines the reward function:
```json
{
  "reward": {"variables": {"score": {"reward": 1.0}}},
  "done": {"variables": {"lives": {"op": "equal", "reference": 0}}}
}
```

This means: reward = change in score; episode ends when lives == 0.

In [ ]:
# Demonstrate creating a retro environment
try:
    import retro

    # List available games (only those with imported ROMs)
    games = retro.data.list_games()
    print(f"Number of games with ROMs available: {len(games)}")
    if games:
        print(f"First 10: {sorted(games)[:10]}")
    print()

    # Try to create a retro environment
    # Pick the first available game or a known one
    test_game = "SuperMarioBros-Nes" if "SuperMarioBros-Nes" in games else (games[0] if games else None)

    if test_game:
        env = retro.make(
            game=test_game,
            use_restricted_actions=retro.Actions.FILTERED,
            render_mode="rgb_array",
        )
        obs, info = env.reset()
        print(f"Game: {test_game}")
        print(f"Observation shape: {obs.shape}  (varies by console)")
        print(f"Action space: {env.action_space}")
        print(f"  → This is MultiBinary: each index is a button (A, B, Up, Down, ...)")

        # Show the raw frame
        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        ax.imshow(obs)
        ax.set_title(f"{test_game} — raw frame\n{obs.shape}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

        env.close()
    else:
        print("No ROMs imported. Import with: python -m retro.import /path/to/roms")

except ImportError:
    print("stable-retro not installed. Install with: pip install stable-retro")
    print("\nKey difference from Atari: retro uses MultiBinary action spaces.")
    print("Each button (A, B, Up, Down, Left, Right, ...) is independently on/off.")

### 3.3 Button-to-MultiBinary Action Space Mapping

Unlike Atari's `Discrete(n)` space where action 3 might mean "move right", retro uses `MultiBinary(n)` where each element corresponds to a physical controller button:

**Genesis controller (6 buttons + D-pad):**
```
Index:  0    1     2      3       4     5      6    7    8    9   10   11
Button: B    A     MODE   START   UP    DOWN   LEFT RIGHT C    Y    X    Z
```

An action like `[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]` means "press B + Right" — e.g., a running attack in a fighting game.

**For 2-player games**, the action space doubles: the first `n` buttons are Player 1, the next `n` are Player 2. So `MultiBinary(24)` for a Genesis 2-player game:

```
indices  0-11: Player 1 buttons
indices 12-23: Player 2 buttons
```

This is important for GOLDS's self-play architecture (Section 5), where we split the action space in half.

### 3.4 Action Space Reduction — DiscreteActionWrapper

The MultiBinary action space gives the agent all possible button combinations — including nonsensical ones like pressing Up+Down simultaneously. For a Genesis game with 12 buttons, that's $2^{12} = 4096$ possible combos. Even with `Actions.FILTERED`, we still get ~126.

This is a huge problem for learning. The agent wastes training exploring useless actions like "press all buttons at once" instead of discovering that "Right+A" means "jump forward."

GOLDS v3 introduces `DiscreteActionWrapper` which maps a small `Discrete(N)` action space to the meaningful button combinations per genre:

| Action set | N | Key actions |
|------------|---|-------------|
| `platformer` | 9 | NOOP, Left, Right, Down, Run, Spin Dash, Jump, Jump Forward, Running Jump |
| `fighter` | 12 | NOOP, directionals, A, B, and common combos |
| `puzzle` | 5 | NOOP, Left, Right, Down, Rotate |

This is the **single highest-impact change** for retro game training, based on findings from the OpenAI Retro Contest. The wrapper reads `env.unwrapped.buttons` to automatically handle NES (8 buttons) vs Genesis (12 buttons).

In [ ]:
# Demonstrate action set mapping
from golds.environments.retro.wrappers import ACTION_SETS

for name, combos in ACTION_SETS.items():
    print(f"\n{name} ({len(combos)} actions):")
    for i, (label, buttons) in enumerate(combos):
        btn_str = "+".join(buttons) if buttons else "(nothing)"
        print(f"  [{i}] {label:15s} → {btn_str}")

### 3.5 Sticky Actions — Breaking Determinism

Retro games are **fully deterministic**: the same sequence of button presses always produces the same outcome. Without any randomization, an RL agent can memorize a specific frame-by-frame action sequence without learning to react to what it sees on screen.

`StickyActionWrapper` fixes this by repeating the previous action with probability $p$ (default 0.25):

$$a_t = \begin{cases} a_{t-1} & \text{with probability } p \\ a_t^{\text{agent}} & \text{with probability } 1-p \end{cases}$$

This forces the agent to build a **reactive policy** that responds to observations rather than replaying memorized sequences. It's standard practice in retro RL research.

In [ ]:
# Simulate sticky action probability over 1000 steps
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
stickprob = 0.25
n_steps = 200

agent_actions = rng.integers(0, 9, size=n_steps)
actual_actions = np.zeros(n_steps, dtype=int)
actual_actions[0] = agent_actions[0]

for t in range(1, n_steps):
    if rng.random() < stickprob:
        actual_actions[t] = actual_actions[t-1]  # sticky!
    else:
        actual_actions[t] = agent_actions[t]

matches = (agent_actions == actual_actions).sum()
print(f"Agent intended action executed: {matches}/{n_steps} ({100*matches/n_steps:.1f}%)")
print(f"Action was sticky (repeated): {n_steps - matches}/{n_steps} ({100*(n_steps-matches)/n_steps:.1f}%)")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(12, 3))
    sticky_mask = agent_actions != actual_actions
    ax.scatter(np.where(~sticky_mask)[0], agent_actions[~sticky_mask], s=3, alpha=0.5, label="Executed as intended")
    ax.scatter(np.where(sticky_mask)[0], actual_actions[sticky_mask], s=8, color="red", alpha=0.7, label="Sticky (repeated previous)")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Action")
    ax.set_title(f"StickyActionWrapper (p={stickprob}): red = action was overridden")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

### 3.6 Multi-Level Rotation

Training on a single level causes the agent to **overfit to that level's layout**. It memorizes where platforms and enemies are rather than learning general platformer skills (jump timing, enemy avoidance).

`MultiLevelWrapper` addresses this by randomly selecting a level on each episode reset:

```python
# Config example:
# environment:
#   levels: [GreenHillZone.Act1, GreenHillZone.Act2, MarbleZone.Act1]
```

With 24 parallel environments, each independently sampling from the level list, the agent sees diverse level layouts at every training step. This is the approach used by the OpenAI Retro Contest winners (Joint PPO).

### 3.4 GOLDS Retro Preprocessing

GOLDS applies similar preprocessing to retro games as the Atari stack, implemented in `src/golds/environments/retro/maker.py`:

```python
class RetroPreprocessing(gym.Wrapper):
    """Grayscale conversion, resize to 84×84, reward clipping."""
    ...

class FrameSkip(gym.Wrapper):
    """Repeat action for N frames, return max of last 2."""
    ...
```

The pipeline for retro is: `retro.make()` → `Monitor` → `FrameSkip` → `RetroPreprocessing` → vectorization.

---
## 4. Vectorized Environments

PPO (Proximal Policy Optimization) needs to collect many environment transitions in parallel. A single environment would be far too slow — the agent would spend most of its time waiting for the emulator. **Vectorized environments** run multiple copies simultaneously.

### 4.1 Why Vectorize?

PPO collects a batch of experience before each gradient update. If each environment runs sequentially:

$$\text{Time per batch} = n_{\text{steps}} \times t_{\text{step}}$$

With $N$ parallel environments:

$$\text{Time per batch} = \frac{n_{\text{steps}}}{N} \times t_{\text{step}} \quad \text{(ideally)}$$

GOLDS typically uses **8-24 parallel environments** (e.g., `n_envs: 24` for Mortal Kombat II).

### 4.2 DummyVecEnv vs SubprocVecEnv

Stable-Baselines3 provides two vectorized environment implementations:

#### `DummyVecEnv` — Sequential, Single-Process
- Steps through each environment **one at a time** in a for-loop
- No multiprocessing overhead
- Best for: **evaluation** (1 env), simple/fast environments, debugging

```python
# Internally, DummyVecEnv does:
for i, env in enumerate(self.envs):
    obs[i] = env.step(actions[i])
```

#### `SubprocVecEnv` — Parallel, Multi-Process
- Each environment runs in its own **subprocess**
- Steps execute truly in parallel (limited by CPU cores)
- Best for: **training** with many environments, CPU-heavy emulation

```python
# Internally, SubprocVecEnv does:
for pipe in self.remotes:
    pipe.send(("step", action))   # send all at once
for pipe in self.remotes:
    obs = pipe.recv()             # collect results
```

| | DummyVecEnv | SubprocVecEnv |
|---|---|---|
| Parallelism | None | True multiprocess |
| Startup time | Instant | Slower (spawns processes) |
| Memory | Shared | Each process has full copy |
| Best for | Eval, debugging | Training |
| GOLDS usage | `create_eval_env()` | `create()` with `n_envs > 1` |

In [ ]:
import time
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

def make_cartpole():
    """Factory function — required by VecEnv constructors."""
    return gym.make("CartPole-v1")

n_envs = 4
n_steps = 500

# --- DummyVecEnv (sequential) ---
dummy = DummyVecEnv([make_cartpole for _ in range(n_envs)])
dummy.reset()

t0 = time.perf_counter()
for _ in range(n_steps):
    actions = np.array([dummy.action_space.sample() for _ in range(n_envs)])
    obs, rewards, dones, infos = dummy.step(actions)
dummy_time = time.perf_counter() - t0
dummy.close()

# --- SubprocVecEnv (parallel) ---
try:
    subproc = SubprocVecEnv([make_cartpole for _ in range(n_envs)])
    subproc.reset()

    t0 = time.perf_counter()
    for _ in range(n_steps):
        actions = np.array([subproc.action_space.sample() for _ in range(n_envs)])
        obs, rewards, dones, infos = subproc.step(actions)
    subproc_time = time.perf_counter() - t0
    subproc.close()

    print(f"DummyVecEnv   ({n_envs} envs, {n_steps} steps): {dummy_time:.3f}s")
    print(f"SubprocVecEnv ({n_envs} envs, {n_steps} steps): {subproc_time:.3f}s")
    print(f"\nNote: For simple envs like CartPole, DummyVecEnv may be faster")
    print(f"due to multiprocessing overhead. The benefit shows with heavy")
    print(f"emulation (Atari/retro) where step() is expensive.")
except Exception as e:
    print(f"SubprocVecEnv failed (common in notebooks): {e}")
    print(f"\nDummyVecEnv ({n_envs} envs, {n_steps} steps): {dummy_time:.3f}s")

### 4.3 VecFrameStack — Temporal Context

A single 84×84 grayscale frame is a snapshot — it tells you where the ball *is*, but not where it's *going*. To give the agent a sense of **motion and velocity**, we stack the last $k$ frames together.

With `frame_stack=4`, the observation shape goes from `(84, 84, 1)` to `(84, 84, 4)` — four consecutive frames stacked along the channel dimension.

This is critical because from stacked frames, a CNN can learn to detect:
- **Direction of motion** — the ball moved left between frames 1 and 2
- **Speed** — the ball moved 5 pixels per frame, so it's fast
- **Acceleration** — the distance between positions is increasing

Without stacking, the agent would need recurrent memory (LSTM) to track motion — stacking is a simpler and very effective alternative.

$$\text{obs}_t = \text{concat}\bigl[f_{t-3},\; f_{t-2},\; f_{t-1},\; f_{t}\bigr] \in \mathbb{R}^{84 \times 84 \times 4}$$

In [ ]:
from stable_baselines3.common.vec_env import VecFrameStack

# Create a simple vectorized env and add frame stacking
base_env = DummyVecEnv([make_cartpole])

print(f"Before stacking: obs_space = {base_env.observation_space}")
print(f"  shape: {base_env.observation_space.shape}")

stacked_env = VecFrameStack(base_env, n_stack=4)

print(f"\nAfter stacking (n_stack=4): obs_space = {stacked_env.observation_space}")
print(f"  shape: {stacked_env.observation_space.shape}")

# Show the stacked observation
obs = stacked_env.reset()
print(f"\nReset observation shape: {obs.shape}")
print(f"  (batch_size=1, features × n_stack = {base_env.observation_space.shape[0]} × 4 = {obs.shape[1]})")

stacked_env.close()

### 4.4 VecTransposeImage — HWC to CHW

Gymnasium environments output images in **HWC** format (Height × Width × Channels), which is what matplotlib and OpenCV expect. But PyTorch's `Conv2d` expects **CHW** format (Channels × Height × Width).

`VecTransposeImage` handles this conversion automatically:

```
Before: (84, 84, 4)  ← H=84, W=84, C=4 (4 stacked grayscale frames)
After:  (4, 84, 84)  ← C=4, H=84, W=84 (PyTorch convention)
```

This is the last step before the observation reaches the neural network.

### 4.5 The Complete GOLDS Pipeline

Here is the full pipeline as implemented in `src/golds/environments/factory.py`:

```
                     Platform-specific preprocessing
                     ┌─────────────────────────────┐
Atari:   gym.make() │ NoopReset → MaxSkip → Life  │ → VecEnv
                     │ → Fire → Warp → ClipReward  │     │
                     └─────────────────────────────┘     │
                                                         │
Retro:   retro.make()                                    │
         │ → Monitor → FrameSkip → RetroPreprocessing    │
         └───────────────────────────────────────→ VecEnv │
                                                         │
                     Common VecEnv wrappers              ▼
                     ┌─────────────────────────────────────┐
                     │ VecFrameStack(n_stack=4)            │
                     │ VecTransposeImage (HWC → CHW)       │
                     │ [Optional] VecNormalize (rewards)   │
                     │ [Optional] VecTwoPlayerOpponent     │
                     └─────────────────────────────────────┘
                                    │
                                    ▼
                        PPO agent receives
                        (N, 4, 84, 84) uint8
```

In [ ]:
# Demonstrate the full GOLDS pipeline (using the project's factory)
try:
    from golds.environments.factory import EnvironmentFactory

    # Create a fully preprocessed vectorized environment
    vec_env = EnvironmentFactory.create(
        game_id="pong",      # or "breakout", "space_invaders", etc.
        n_envs=2,
        frame_stack=4,
        seed=42,
        use_subproc=False,   # DummyVecEnv for notebook compatibility
    )

    print(f"Observation space: {vec_env.observation_space}")
    print(f"  shape: {vec_env.observation_space.shape}  ← (C, H, W) = (4 stacked, 84, 84)")
    print(f"  dtype: {vec_env.observation_space.dtype}")
    print(f"Action space: {vec_env.action_space}")
    print(f"Number of envs: {vec_env.num_envs}")

    obs = vec_env.reset()
    print(f"\nBatch observation shape: {obs.shape}  ← (n_envs, C, H, W)")

    vec_env.close()

except Exception as e:
    print(f"Could not create GOLDS environment: {e}")
    print("\nExpected output shape: (n_envs, 4, 84, 84) uint8")
    print("  - n_envs: number of parallel environments")
    print("  - 4: frame stack depth (temporal context)")
    print("  - 84×84: preprocessed grayscale frame")

---
## 5. Self-Play Architecture

Some games in GOLDS are **two-player** — most notably Mortal Kombat II. Training an agent against a static (or random) opponent leads to **overfitting to that opponent's strategy**. Self-play solves this by having the agent train against past versions of itself.

### 5.1 The Problem with Fixed Opponents

If the opponent always does the same thing, the agent learns a **narrow counter-strategy** rather than general fighting skills. This is analogous to only studying one textbook for an exam — you might ace that specific textbook's questions but fail on anything else.

Self-play provides a **curriculum** that automatically adapts in difficulty:
- Early training: opponent is random/weak → agent learns basics
- Mid training: opponent uses earlier (weaker) policies → agent refines strategy
- Late training: opponent uses recent (strong) policies → agent must innovate

### 5.2 VecTwoPlayerOpponentWrapper — Architecture

GOLDS implements self-play with `VecTwoPlayerOpponentWrapper` (defined in `src/golds/environments/retro/self_play.py`). Here is the architecture:

```
                    ┌─────────────────────────────────────────┐
                    │       VecTwoPlayerOpponentWrapper        │
                    │                                         │
   PPO Agent        │  ┌───────────┐     ┌───────────────┐   │
   (learner)        │  │ P1 action │     │ Opponent      │   │
      │             │  │ (from PPO)│     │ Policy        │   │
      │  action     │  └─────┬─────┘     │ (snapshot)    │   │
      └──────────►  │        │            └──────┬────────┘   │
                    │        │   concatenate     │            │
                    │        └──────┬─────────────┘            │
                    │               │                          │
                    │               ▼                          │
                    │     ┌─────────────────┐                  │
                    │     │ Retro Env       │                  │
                    │     │ MultiBinary(24) │                  │
                    │     │ [P1_btns|P2_btns]│                 │
                    │     └────────┬────────┘                  │
                    │              │                           │
                    │       obs, reward                        │
      ◄────────────│◄──────────────┘                           │
   PPO receives     │                                         │
   (obs, reward)    └─────────────────────────────────────────┘
```

The wrapper:
1. Receives a **Player 1 action** from PPO (size `n_buttons // 2`)
2. Generates a **Player 2 action** from the opponent policy
3. **Concatenates** `[P1_action | P2_action]` into the full `MultiBinary(2 * n_buttons)` action
4. Steps the underlying environment with the combined action
5. Returns only the observation and reward (from Player 1's perspective) to PPO

From PPO's point of view, it's playing a single-player game with `MultiBinary(n_buttons // 2)` actions.

### 5.3 Opponent Modes

GOLDS supports several opponent modes, controlled by the `OpponentSpec` dataclass:

| Mode | P2 Action | Use Case |
|------|-----------|----------|
| `"none"` / `"noop"` | All zeros (do nothing) | Baseline, early training |
| `"random"` | Random button presses | Slightly harder baseline |
| `"model"` | Fixed model loaded from disk | Evaluation against specific opponent |
| `"self_play"` | Periodically reloaded from snapshot directory | **Full self-play training** |

```python
@dataclass
class OpponentSpec:
    mode: OpponentMode = "none"           # "none", "random", "model", "self_play"
    model_path: Path | None = None         # fixed model for "model" mode
    snapshot_dir: Path | None = None       # directory of snapshots for "self_play"
    reload_interval_steps: int = 500       # how often to check for new snapshots
```

### 5.4 Snapshot Lifecycle

During self-play training, the snapshot lifecycle works as follows:

```
Timeline ──────────────────────────────────────────────────────►

Training:    ████████████████████████████████████████████████████

Snapshots:        ▼ save           ▼ save           ▼ save
              opponent_500k.zip  opponent_1M.zip  opponent_1.5M.zip
                    │                │                │
Opponent pool:      └────────────────┴────────────────┘
                    [500k, 1M, 1.5M]  ← latest selected
                           │
Wrapper reloads:    ──▼──────▼──────▼──────▼──────▼──────
                    (every reload_interval_steps)
```

**Step by step:**

1. **Train:** PPO trains the agent (Player 1) against the current opponent
2. **Snapshot:** Every `self_play_snapshot_freq` timesteps, save the current model as `opponent_<timesteps>.zip`
3. **Opponent Pool:** The snapshot directory accumulates past model versions (capped at `self_play_max_snapshots`)
4. **Sample:** The wrapper periodically checks the snapshot directory and loads the **latest** valid snapshot as the new opponent

From the Mortal Kombat II config:
```yaml
training:
  self_play_snapshot_freq: 500000    # save snapshot every 500k steps
  self_play_max_snapshots: 5         # keep at most 5 past versions
```

In [ ]:
# Visualize the self-play snapshot lifecycle
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

# Training progress
timesteps = np.linspace(0, 30, 300)  # 30M steps
skill = 1 - np.exp(-0.15 * timesteps)  # hypothetical skill curve
ax.plot(timesteps, skill, 'b-', linewidth=2, label='Agent skill (learner)')

# Snapshot points
snapshot_times = np.arange(0.5, 30, 0.5)  # every 500k = 0.5M
snapshot_skill = 1 - np.exp(-0.15 * snapshot_times)

# Only show last 5 snapshots at each point (max_snapshots=5)
# For visualization, show all snapshot events
ax.scatter(snapshot_times[::2], snapshot_skill[::2], 
           color='red', s=40, zorder=5, marker='v', label='Snapshot saved')

# Opponent skill (step function, lagging behind agent)
opponent_skill = np.zeros_like(timesteps)
for i, t in enumerate(timesteps):
    # Opponent uses latest snapshot that exists
    past = snapshot_times[snapshot_times <= t]
    if len(past) > 0:
        opp_t = past[-1]
        opponent_skill[i] = 1 - np.exp(-0.15 * opp_t)

ax.step(timesteps, opponent_skill, 'r--', linewidth=1.5, alpha=0.7, 
        label='Opponent skill (latest snapshot)', where='post')

# Shade the gap
ax.fill_between(timesteps, opponent_skill, skill, alpha=0.1, color='blue',
                label='Skill gap (drives learning)')

ax.set_xlabel('Training timesteps (millions)', fontsize=12)
ax.set_ylabel('Relative skill', fontsize=12)
ax.set_title('Self-Play: Agent vs. Past Versions of Itself', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0, 30)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Show the action space splitting in practice
print("=== Action Space Splitting for 2-Player Games ===")
print()

# Simulate a Genesis controller layout (12 buttons per player)
genesis_buttons = ["B", "A", "MODE", "START", "UP", "DOWN", "LEFT", "RIGHT", "C", "Y", "X", "Z"]
n_buttons = len(genesis_buttons)

print(f"Genesis controller: {n_buttons} buttons per player")
print(f"2-player action space: MultiBinary({2 * n_buttons})")
print()

# Show the split
print("Full action vector indices:")
print(f"  Player 1: indices 0-{n_buttons-1}")
for i, btn in enumerate(genesis_buttons):
    print(f"    [{i:2d}] {btn}")
print(f"  Player 2: indices {n_buttons}-{2*n_buttons-1}")
for i, btn in enumerate(genesis_buttons):
    print(f"    [{i+n_buttons:2d}] {btn}")

print()
print("VecTwoPlayerOpponentWrapper exposes only P1 actions:")
print(f"  Learner sees: MultiBinary({n_buttons})")
print(f"  Wrapper appends P2 actions from opponent policy")
print(f"  Underlying env receives: MultiBinary({2*n_buttons})")

In [ ]:
# Demonstrate opponent action generation for different modes
n_envs = 4
buttons_per_player = 12

print("=== Opponent Action Generation ===")
print(f"(n_envs={n_envs}, buttons_per_player={buttons_per_player})")
print()

# Noop mode
noop_actions = np.zeros((n_envs, buttons_per_player), dtype=np.int8)
print("Mode: 'noop'")
print(f"  P2 actions (all zeros):\n  {noop_actions}")
print()

# Random mode
np.random.seed(42)
random_actions = np.random.randint(0, 2, size=(n_envs, buttons_per_player), dtype=np.int8)
print("Mode: 'random'")
print(f"  P2 actions (random buttons):\n  {random_actions}")
print()

# Concatenation example
p1_actions = np.zeros((n_envs, buttons_per_player), dtype=np.int8)
p1_actions[0, 7] = 1  # env 0: press RIGHT
p1_actions[0, 0] = 1  # env 0: press B (punch)
combined = np.concatenate([p1_actions, random_actions], axis=1)

print("Combined action (P1 + P2):")
print(f"  Shape: {combined.shape}  ← ({n_envs}, {2 * buttons_per_player})")
print(f"  Env 0: {combined[0]}")
print(f"         {'P1 buttons':^{3*buttons_per_player}}  {'P2 buttons':^{3*buttons_per_player}}")

### 5.5 Why Self-Play Works

Self-play creates an **emergent curriculum**. The key insight is from game theory:

In a two-player zero-sum game, the optimal strategy is a **Nash equilibrium** — a policy where neither player can improve by changing strategy unilaterally. Self-play approximates this through iterative best-response:

1. Start with policy $\pi_0$ (random)
2. Train $\pi_1$ to beat $\pi_0$
3. Train $\pi_2$ to beat $\pi_1$
4. ...

Under certain conditions, this sequence converges to a Nash equilibrium. In practice (with PPO and neural networks), it produces strong, robust policies even if true convergence is not guaranteed.

GOLDS uses a simplified version: the opponent is always the **latest** snapshot, not a mixture of past policies. This is simpler to implement and works well in practice for games like Mortal Kombat II, where the skill gap between consecutive snapshots provides a consistent learning signal.

---
## Summary

The GOLDS v3 retro environment pipeline applies wrappers in this order:

```
retro.make() → DiscreteAction → StickyAction → MultiLevel → PlatformerReward → TimeLimit → Monitor → FrameSkip → RetroPreprocessing → VecEnv → VecFrameStack → VecTransposeImage → [RND] → [SelfPlay]
```

| Stage | What Happens | Why |
|-------|-------------|-----|
| **Raw Env** | `gym.make()` / `retro.make()` | Create the base environment |
| **DiscreteAction** | Map `Discrete(N)` to meaningful button combos | Reduce action space from 4096 to 5-12 |
| **StickyAction** | Repeat previous action with probability *p* | Break determinism, force reactive policy |
| **MultiLevel** | Random level selection on reset | Prevent overfitting to one layout |
| **PlatformerReward** | Custom reward shaping per genre | Better learning signal |
| **TimeLimit** | Truncate episode after max steps | Prevent stuck episodes |
| **Monitor** | Log episode stats | Tracking |
| **FrameSkip** | Repeat action 4x, max-pool last 2 frames | Speed + flicker removal |
| **RetroPreprocessing** | Grayscale + resize to 84x84 | 93% input reduction |
| **VecEnv** | N parallel copies | Faster data collection |
| **VecFrameStack** | Stack last 4 frames | Temporal context (motion) |
| **VecTransposeImage** | HWC → CHW | PyTorch convention |
| **[RND]** | Random Network Distillation (optional) | Exploration bonus |
| **[SelfPlay]** | Opponent from past snapshots (optional) | Adaptive difficulty |

The final observation that PPO sees: **(N, 4, 84, 84)** — N parallel environments, 4 stacked grayscale frames, each 84×84 pixels.

**Next notebook:** We will look at the PPO algorithm itself and how it uses these observations to learn a policy.